# 第 5 章 · Tool Use（CrewAI）

同一件事换个框架：`@tool` 的角色一样，变的是**谁来驱动它**——
CrewAI 把编排拆成三个显式对象：`Agent`（谁做）、`Task`（做什么）、`Crew`（怎么组织）。

**kernel 选 `Python (agentic-crewai)`**（独立 venv，已装 crewai 1.15.34，不动主环境）。

这份**尚未执行**，留给你自己跑。已验证可用，跑的时候会看到一条 litellm 的
"approximate input token count" 警告——DeepSeek 不在它的内置 token 表里，无害。

它值得跑一次的原因：在 LangChain 版你能看到 `Invoking: ... with {...}` 这种轨迹，
这里循环被 `Crew` 收走了，正好对比"还剩多少可见性"。


In [ ]:
# pip install crewai
import os
import threading

from crewai import Agent, Task, Crew
from crewai.tools import tool          # ★ 本章新增（CrewAI 版）：与 LangChain 的 @tool 同角色

# 【适配】CrewAI 走 litellm，读 OPENAI_API_KEY / OPENAI_API_BASE；本仓库 .env 已指向 DeepSeek
os.environ.setdefault("OPENAI_MODEL_NAME", "openai/deepseek-flash")


@tool("Stock Price Lookup Tool")
def get_stock_price(ticker: str) -> float:
    """
    Fetches the latest simulated stock price for a given stock ticker symbol.
    Returns the price as a float. Raises a ValueError if the ticker is not found.
    """
    simulated_prices = {"AAPL": 178.15, "GOOGL": 1750.30, "MSFT": 425.50}
    price = simulated_prices.get(ticker.upper())
    if price is not None:
        return price
    # ★ 工具内部抛标准异常，而不是返回一句人话 —— 否则 agent 分不清"拿到结果"和"出错了"
    raise ValueError(f"Simulated price for ticker '{ticker.upper()}' not found.")


# ★ Agent / Task / Crew 三层：这是 CrewAI 与 LangChain 的主要差别（第 7 章展开）
financial_analyst_agent = Agent(
    role='Senior Financial Analyst',
    goal='Analyze stock data using provided tools and report key prices.',
    backstory="You are an experienced financial analyst adept at using data sources to find stock information. You provide clear, direct answers.",
    verbose=True,
    tools=[get_stock_price],
    allow_delegation=False,
    # allow_delegation=True,
)

analyze_aapl_task = Task(
    description=(
        "What is the current simulated stock price for Apple (ticker: AAPL)? "
        "Use the 'Stock Price Lookup Tool' to find it. "
        "If the ticker is not found, you must report that you were unable to retrieve the price."
    ),
    expected_output=(
        "A single, clear sentence stating the simulated stock price for AAPL. "
        "For example: 'The simulated stock price for AAPL is $178.15.' "
        "If the price cannot be found, state that clearly."
    ),
    # expected_output='Output MUST be a JSON object with keys "price" and "currency".',
    agent=financial_analyst_agent,
)

financial_crew = Crew(
    agents=[financial_analyst_agent],
    tasks=[analyze_aapl_task],
    verbose=True,
)


# ★ 框架把循环收走后，verbose 几乎什么都不打（1.15 实测只剩一行 [Finalize]）。
#   要拿回轨迹得订阅事件总线 —— 这正是本章"可观测性"那条线的落点：
#   LangChain 循环是你写的，日志天生可见；CrewAI 要主动订阅；
#   而 google_search 那种厂商侧执行，订阅了也拿不到。
from crewai.events.event_bus import crewai_event_bus
from crewai.events.event_types import ToolUsageStartedEvent, ToolUsageFinishedEvent


@crewai_event_bus.on(ToolUsageStartedEvent)
def _on_tool_start(source, event):
    print(f"   [事件] 工具调用: {getattr(event, 'tool_name', '?')} args={getattr(event, 'tool_args', None)}")


@crewai_event_bus.on(ToolUsageFinishedEvent)
def _on_tool_end(source, event):
    print(f"   [事件] 工具返回: {str(getattr(event, 'output', ''))[:100]}")


# 【适配】VS Code 的 kernel 里本身有一个事件循环在跑，CrewAI 的同步 kickoff() 会检测到
#   并直接抛 RuntimeError（"invoked synchronously from within a running event loop"）。
#   它给的解法是 kickoff_async()，但那要改写成协程。这里放进独立线程更省事：
#   线程里没有运行中的循环，同步接口就能正常工作。
def run_in_thread(fn):
    box = {}

    def _target():
        try:
            box["result"] = fn()
        except BaseException as e:               # noqa: BLE001
            box["error"] = e

    th = threading.Thread(target=_target)
    th.start()
    th.join()
    if "error" in box:
        raise box["error"]
    return box["result"]


def main():
    if not os.environ.get("OPENAI_API_KEY"):
        print("ERROR: OPENAI_API_KEY 未设置（本仓库由 .env 提供）")
        return
    print("\n## Starting the Financial Crew...")
    result = run_in_thread(financial_crew.kickoff)
    print("\n## Crew execution finished.\nFinal Result:\n", result)


if __name__ == "__main__":
    main()



## Starting the Financial Crew...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2835f07f-ebd1-4d60-bfe9-c41cba679c53                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: What is the current simulated stock price for Apple (ticker: AAPL)? Use the 'Stock Price Lookup Tool'    │
│  to find it. If the ticker is not found, you must report that you were unable to retrieve the price.            │
│  ID: 8b9ae5b3-521f-4758-9657-be0fe14e7fa0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Financial Analyst                                                                                │
│                                                                                                                 │
│  Task: What is the current simulated stock price for Apple (ticker: AAPL)? Use the 'Stock Price Lookup Tool'    │
│  to find it. If the ticker is not found, you must report that you were unable to retrieve the price.            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   [事件] 工具调用: stock_price_lookup_tool args={'ticker': 'AAPL'}   [事件] 工具返回: 178.15
   [事件] 工具返回: 178.15
   [事件] 工具返回: 178.15
   [事件] 工具返回: 178.15


╭─────────────────────────────────────── ✅ Tool Execution Completed (#3) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: stock_price_lookup_tool                                                                                  │
│  Output: 178.15                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (3 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: What is the current simulated stock price for Apple (ticker: AAPL)? Use the 'Stock Price Lookup Tool'    │
│  to find it. If the ticker is not found, you must report that you were unable to retrieve the price.            │
│  Agent: Senior Financial Analyst                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


## Crew execution finished.
Final Result:
 {"price": 178.15, "currency": "USD"}


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 2835f07f-ebd1-4d60-bfe9-c41cba679c53                                                                       │
│  Final Output: {"price": 178.15, "currency": "USD"}                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯